# 04 — Validation: the Alaska–Hawaiian merger

Alaska Air Group acquired Hawaiian Holdings on 18 September 2024. We test whether the demand models,
estimated **only on pre-merger data** (2022Q1–2024Q2), predict the fare changes that followed.

1. **Prediction.** Re-estimate the demand models on pre-merger quarters. In each of the last three pre-merger
   quarters (2023Q4, 2024Q1, 2024Q2), recover marginal costs from the Bertrand–Nash first-order conditions,
   combine Alaska (AS) and Hawaiian (HA) ownership and solve for the new equilibrium. The route-level prediction
   is the passenger-weighted change in the merging carriers' fares, averaged over the three base quarters.
2. **Outcome.** Realised fare changes by difference in differences: same seasons one year apart
   (2023Q4–2024Q2 vs 2024Q4–2025Q2) on overlap routes, relative to control routes served by neither carrier.
   The closing quarter 2024Q3 is excluded.
3. **Comparison.** Average prediction against the DiD estimate, and route-by-route errors.

In [ ]:
%run 00_setup.ipynb

In [ ]:
%run 00_model_functions.ipynb

In [ ]:
PARTIES = ("AS", "HA")
BASE = [(2023, 4), (2024, 1), (2024, 2)]
POST = AS_HA_POST
WEST = {"WA", "OR", "CA", "NV", "AZ", "ID", "UT", "MT", "WY", "CO", "NM", "AK", "HI"}
IV = IV_SETS["baseline"]
base_ids = {qlabel(*q) for q in BASE}

p_all = pd.read_parquet(INTERIM / "products_airport_nondir_iv.parquet")
est_full = load_json("demand_full_sample")
pre = p_all[p_all.t <= qindex(*AS_HA_PRE[-1])].sort_values(["market_ids", "carrier", "nonstop"]).reset_index(drop=True)
print(f"pre-merger sample: {len(pre):,} products, {pre.market_ids.nunique():,} markets")

## Overlap routes

In [ ]:
mk_both = pre[pre.quarter.isin(base_ids)].groupby("market_ids").carrier.agg(lambda s: set(PARTIES) <= set(s))
sim_markets = mk_both.index[mk_both].tolist()
overlap_routes = sorted({m.split("|")[0] for m in sim_markets})
print(len(sim_markets), "overlap markets in the base quarters;", len(overlap_routes), "overlap routes")
ov = pre[pre.market_ids.isin(sim_markets)]
ov_summary = ov.groupby("route").apply(lambda d: pd.Series({
    "AS share": d.loc[d.carrier == "AS", "passengers"].sum() / d.passengers.sum(),
    "HA share": d.loc[d.carrier == "HA", "passengers"].sum() / d.passengers.sum(),
    "AS nonstop": int((d.carrier.eq("AS") & d.nonstop.eq(1)).any()),
    "HA nonstop": int((d.carrier.eq("HA") & d.nonstop.eq(1)).any()),
    "carriers": d.carrier.nunique(), "passengers (000/qtr)": d.passengers.sum() / 3 / 1e3})).sort_values(
    "passengers (000/qtr)", ascending=False)
ov_summary.head(20)

## Pre-merger demand estimates

Linear models are re-estimated on the pre-merger panel. The random-coefficients model (demand only, the
preferred specification of notebook 03: the joint demand-supply fit stops at a saddle point) is re-estimated on the pre-merger quarters of the same sample of routes,
starting from the full-sample estimates.

In [ ]:
v_logit = linear_iv(pre, IV)
v_nls = linear_iv(pre, IV, nest="service")
v_nla = linear_iv(pre, IV, nest="air")
rc_routes = est_full["rc_routes"][: (120 if SMOKE else None)]
pre_rc = pre[pre.route.isin(rc_routes)].reset_index(drop=True)
prob = make_problem(pre_rc, "rc", IV)
with pyblp.parallel(N_CORES):
    v_rc = solve_rc(prob, "rc", sigma0=est_full["rc_demand"]["sigma"], pi0=est_full["rc_demand"]["pi"], method="2s")
print(v_rc)
est = pd.DataFrame({
    "Logit": {"alpha": -v_logit["alpha"], "alpha s.e.": v_logit["alpha_se"]},
    "NL (nonstop/connecting)": {"alpha": -v_nls["alpha"], "alpha s.e.": v_nls["alpha_se"], "rho": v_nls["rho"], "rho s.e.": v_nls["rho_se"]},
    "NL (air travel)": {"alpha": -v_nla["alpha"], "alpha s.e.": v_nla["alpha_se"], "rho": v_nla["rho"], "rho s.e.": v_nla["rho_se"]},
    "RC demand": {"pi_0": float(v_rc.pi[1, 0]), "pi_0 s.e.": float(v_rc.pi_se[1, 0]),
                         **{f"sigma {l}": float(v_rc.sigma[i, i]) for i, l in enumerate(["constant", "price", "nonstop"])}}}).astype(float)
save_table(est, "asha_pre_estimates", float_format="%.3f", note="Demand estimated on 2022Q1--2024Q2 only.")
est

## Merger simulation in the base quarters

In [ ]:
sim_sample = pre[pre.quarter.isin(base_ids)].reset_index(drop=True)
MODELS = {"Logit": ("logit", {"alpha": v_logit["alpha"]}),
          "NL (nonstop/connecting)": ("nested_service", {"alpha": v_nls["alpha"], "rho": v_nls["rho"]}),
          "NL (air travel)": ("nested_air", {"alpha": v_nla["alpha"], "rho": v_nla["rho"]}),
          "RC demand": ("rc", {"sigma": np.diag(v_rc.sigma).tolist(), "pi": np.asarray(v_rc.pi).tolist()})}
pred, sims, results = {}, {}, {}
for name, (kind, prm) in MODELS.items():
    with pyblp.parallel(N_CORES):
        r = evaluate_fixed(sim_sample, kind, prm, IV)
    results[name] = r
    sim, cs = simulate_merger(r, sim_sample, PARTIES, markets=sim_markets)
    rs = route_summary(sim, cs, PARTIES)
    sims[name] = (sim, rs)
    pred[name] = rs.groupby("route").apply(lambda d: pd.Series({
        "pred_parties": np.average(d.dprice_parties, weights=d.passengers),
        "pred_rivals": np.average(d.dprice_rivals.fillna(0), weights=d.passengers),
        "dcs_usd_q": d.dcs_usd.mean()}))
    print(name, "mean predicted AS/HA fare change (%):", round(party_avg(rs), 2))

## Realised fare changes: difference in differences

*Treated products*: AS and HA products on overlap routes. *Controls*: products on routes with an endpoint in a
western state where neither AS nor HA sold tickets in any quarter. A second control group uses all unaffected
routes. Product fixed effects (carrier × route × service type) hold composition fixed; quarter fixed effects
absorb common shocks and seasonality. Weighted by passengers; standard errors clustered by route.

In [ ]:
apt_state = pd.read_parquet(CLEAN / "airport_cbsa.parquet").set_index("AIRPORT").AIRPORT_STATE_CODE
panel = p_all[~p_all.quarter.isin([qlabel(*q) for q in AS_HA_TRANSITION])].copy()
panel["product"] = panel.route + ":" + panel.carrier + ":" + panel.nonstop.astype(str)
party_routes = set(panel.loc[panel.carrier.isin(PARTIES), "route"])
panel["west"] = panel.a.map(apt_state).isin(WEST) | panel.b.map(apt_state).isin(WEST)
panel["post"] = (panel.t >= qindex(*POST[0])).astype(int)
panel["overlap"] = panel.route.isin(overlap_routes)
panel["party"] = panel.carrier.isin(PARTIES)
panel["lfare"] = np.log(panel.fare)
ctrl_west = (~panel.route.isin(party_routes)) & panel.west
ctrl_all = ~panel.route.isin(party_routes)

def did(sample, treat, label, window=None, y="lfare"):
    d = panel[sample | treat].copy()
    if window is not None:
        d = d[d.t.isin(window)]
    d["treat_post"] = (treat[d.index] & (d.post == 1)).astype(float)
    absorb = pd.DataFrame({"product": d["product"].astype("category"), "quarter": d.quarter.astype("category")})
    f = AbsorbingLS(d[y], d[["treat_post"]], absorb=absorb, weights=d.passengers) \
        .fit(cov_type="clustered", clusters=d.route.astype("category").cat.codes)
    b, se = f.params["treat_post"], f.std_errors["treat_post"]
    return {"specification": label, "coef (log points)": b, "s.e.": se, "% change": 100 * (np.exp(b) - 1),
            "95% CI low": 100 * (np.exp(b - 1.96 * se) - 1), "95% CI high": 100 * (np.exp(b + 1.96 * se) - 1),
            "treated products": int(d.loc[treat[d.index], "product"].nunique()), "N": int(f.nobs)}

treat_parties = panel.overlap & panel.party
treat_rivals = panel.overlap & ~panel.party
win = list(qindex(*np.array(BASE).T)) + list(qindex(*np.array(POST).T))
did_rows = [
    did(ctrl_west, treat_parties, "AS/HA on overlap routes vs western controls (all quarters)"),
    did(ctrl_west, treat_parties, "AS/HA on overlap routes vs western controls (same seasons, 1 year apart)", window=win),
    did(ctrl_all, treat_parties, "AS/HA on overlap routes vs all unaffected routes"),
    did(ctrl_west, treat_rivals, "Rivals on overlap routes vs western controls"),
    did(ctrl_west, panel.party & ~panel.overlap, "AS/HA on non-overlap routes vs western controls"),
]
did_tab = pd.DataFrame(did_rows).set_index("specification")
save_table(did_tab, "asha_did", float_format="%.3f",
           note="Dependent variable: log average fare of the product. Product and quarter fixed effects, passenger weights, "
                "route-clustered standard errors. Post period 2024Q4--2025Q2; 2024Q3 (closing quarter) excluded.")
did_tab

### Event study

In [ ]:
def event_study(sample, treat, ref=qindex(2024, 2)):
    d = panel[sample | treat].copy()
    cols = []
    for t in sorted(d.t.unique()):
        if t == ref:
            continue
        c = f"q{t}"
        d[c] = (treat[d.index] & (d.t == t)).astype(float)
        cols.append(c)
    absorb = pd.DataFrame({"product": d["product"].astype("category"), "quarter": d.quarter.astype("category")})
    f = AbsorbingLS(d.lfare, d[cols], absorb=absorb, weights=d.passengers) \
        .fit(cov_type="clustered", clusters=d.route.astype("category").cat.codes)
    out = pd.DataFrame({"t": [int(c[1:]) for c in cols], "coef": f.params[cols].values, "se": f.std_errors[cols].values})
    return pd.concat([out, pd.DataFrame({"t": [ref], "coef": [0.0], "se": [0.0]})]).sort_values("t")

es = event_study(ctrl_west, treat_parties)
es_r = event_study(ctrl_west, treat_rivals)
es.to_csv(RES / "asha_event_study.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 3.4))
qlab = {qindex(y, q): qlabel(y, q) for (y, q) in QUARTERS}
for df, lab, c, off in [(es, "AS/HA products", PALETTE[0], -0.1), (es_r, "Rival products", PALETTE[1], 0.1)]:
    ax.errorbar(df.t + off, 100 * df.coef, yerr=196 * df.se, fmt="o", color=c, capsize=2, label=lab, ms=4)
ax.axhline(0, color="k", lw=0.8)
ax.axvline(qindex(2024, 3), color="gray", ls="--", lw=0.8)
ax.set_xticks(sorted(qlab)); ax.set_xticklabels([qlab[k] for k in sorted(qlab)], rotation=45)
ax.set_ylabel("Fare relative to 2024Q2 (log points × 100)")
ax.legend(); fig.tight_layout(); savefig(fig, "asha_event_study")
pre_trend = es[es.t < qindex(2024, 2)]
print("joint pre-period: mean coefficient %.3f, max |t| %.2f" % (pre_trend.coef.mean(), (pre_trend.coef / pre_trend.se.replace(0, np.nan)).abs().max()))

## Route-level realised changes and prediction errors

For each overlap route, the realised change is the change in the passenger-weighted mean log fare of the AS/HA
products present in both windows (base quarters vs the same seasons one year later), minus the same change
averaged over western control products.

In [ ]:
def window_change(d):
    pre_w = d[d.quarter.isin(base_ids)]
    post_w = d[d.quarter.isin({qlabel(*q) for q in POST})]
    a = pre_w.groupby("product").apply(lambda g: pd.Series({"l0": np.average(g.lfare, weights=g.passengers),
                                                            "w": g.passengers.sum(), "route": g.route.iat[0]}))
    b = post_w.groupby("product").apply(lambda g: np.average(g.lfare, weights=g.passengers)).rename("l1")
    x = a.join(b, how="inner")
    x["dl"] = x.l1 - x.l0
    return x

chg_t = window_change(panel[treat_parties])
chg_c = window_change(panel[ctrl_west])
ctrl_mean = np.average(chg_c.dl, weights=chg_c.w)
actual = chg_t.groupby("route").apply(lambda g: pd.Series({"actual_dl": np.average(g.dl, weights=g.w), "w": g.w.sum()}))
actual["actual_pct"] = 100 * (np.exp(actual.actual_dl - ctrl_mean) - 1)
print(f"control routes: mean change {100*(np.exp(ctrl_mean)-1):.2f}% over one year ({len(chg_c):,} products)")

cmp_rows, route_tables = [], {}
for name in pred:
    m = actual.join(pred[name], how="inner")
    route_tables[name] = m
    err = m.pred_parties - m.actual_pct
    w = m.w
    cmp_rows.append({
        "model": name, "routes": len(m),
        "mean predicted (%)": np.average(m.pred_parties, weights=w),
        "mean realised, DiD (%)": np.average(m.actual_pct, weights=w),
        "mean error (pp)": np.average(err, weights=w),
        "MAE (pp)": np.average(np.abs(err), weights=w),
        "RMSE (pp)": np.sqrt(np.average(err ** 2, weights=w)),
        "correlation": np.corrcoef(m.pred_parties, m.actual_pct)[0, 1] if len(m) > 2 else np.nan,
        "slope (realised on predicted)": np.polyfit(m.pred_parties, m.actual_pct, 1, w=np.sqrt(w))[0] if len(m) > 2 else np.nan,
    })
cmp_tab = pd.DataFrame(cmp_rows).set_index("model")
save_table(cmp_tab, "asha_prediction_errors", float_format="%.2f",
           note="Predicted changes: passenger-weighted AS/HA fare changes from merger simulation in 2023Q4--2024Q2. Realised: "
                "route-level difference-in-differences changes (same seasons one year apart, net of western control routes). "
                "Errors are predicted minus realised; weights are pre-merger AS/HA passengers. The DiD regression estimate "
                f"(same seasons) is {did_tab.iloc[1]['% change']:.2f}\\% ({did_tab.iloc[1]['95% CI low']:.2f}, {did_tab.iloc[1]['95% CI high']:.2f}).")
cmp_tab

In [ ]:
fig, axes = plt.subplots(1, len(route_tables), figsize=(3.4 * len(route_tables), 3.4), sharex=True, sharey=True)
for ax, (name, m) in zip(axes, route_tables.items()):
    ax.scatter(m.pred_parties, m.actual_pct, s=10 + 60 * m.w / m.w.max(), alpha=0.6, color=PALETTE[0])
    lim = [min(m.pred_parties.min(), m.actual_pct.min(), -10), max(m.pred_parties.max(), m.actual_pct.max(), 10)]
    ax.plot(lim, lim, color="k", lw=0.8, ls="--")
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(name, fontsize=9); ax.set_xlabel("Predicted change (%)")
axes[0].set_ylabel("Realised change, DiD (%)")
fig.tight_layout(); savefig(fig, "asha_predicted_vs_actual")

In [ ]:
top = route_tables["NL (air travel)"][["w", "actual_pct"]].join(ov_summary[["AS share", "HA share", "AS nonstop", "HA nonstop"]])
for name in MODELS:
    top[name] = route_tables[name].pred_parties
top = top.sort_values("w", ascending=False).head(15).drop(columns="w").rename(columns={"actual_pct": "realised (DiD)"})
save_table(top, "asha_routes", float_format="%.2f",
           note="Fifteen largest overlap routes by pre-merger AS/HA passengers: base-quarter shares, nonstop service, predicted "
                "AS/HA fare change by model and realised DiD change (percent).")
top

In [ ]:
if True:
    save_json({"did": did_tab.reset_index().to_dict("records"), "comparison": cmp_tab.reset_index().to_dict("records"),
               "estimates_pre": est.to_dict(), "n_overlap_routes": len(overlap_routes),
               "control_mean_pct": float(100 * (np.exp(ctrl_mean) - 1)),
               "pre_params": {"logit": {"alpha": v_logit["alpha"]}, "nested_service": {"alpha": v_nls["alpha"], "rho": v_nls["rho"]},
                              "nested_air": {"alpha": v_nla["alpha"], "rho": v_nla["rho"]}, "rc": rc_params(v_rc)}}, "asha_validation")
    pd.concat({k: v for k, v in route_tables.items()}).to_csv(RES / "asha_route_predictions.csv")